In [ ]:
import re
import pandas as pd
import os

FILES = {
    ("x86", "o0"): "../data/x86_64_o0.ll",
    ("x86", "o3"): "../data/x86_64_o3.ll",
    ("aarch64", "o0"): "../data/aarch64_o0.ll",
    ("aarch64", "o3"): "../data/aarch64_o3.ll",
}

OP_SUFFIX = {
    "encrypt": "encrypt_block_64_128",
    "encrypt_inflight": "encrypt_block_inflight_64_128",
    "decrypt": "decrypt_block_64_128",
}
SPEC = []
for grp in ["scalar", "sse2", "avx2", "avx512", "neon"]:
    for op, suffix in OP_SUFFIX.items():
        SPEC.append((grp, op, f"{grp}_{suffix}"))

TARGETS = [name for _, _, name in SPEC]

HEAP_RE = re.compile(r"@(__rust_(?:alloc|realloc|dealloc|alloc_zeroed))\b")
ARRAY_RE = re.compile(r"alloca\s+\[(\d+)\s+x\s+i8\]")
ALLOC_SIZE_RE = re.compile(r"@__rust_alloc(?:_zeroed)?\(i64 (\d+)")
REALLOC_SIZE_RE = re.compile(r"@__rust_realloc\(.*i64 (\d+)\s*\)")


def heap_bytes(line):
    m = ALLOC_SIZE_RE.search(line) or REALLOC_SIZE_RE.search(line)
    return int(m.group(1)) if m else 0


In [ ]:
def parse_ir(path):
    lines = open(path).read().splitlines()
    found = {}
    raw_rows = []
    pending, i, n = [], 0, len(lines)

    while i < n:
        line = lines[i];
        st = line.strip()
        if st == "":
            pending = []
        elif st.startswith(";"):
            pending.append(st.lstrip("; ").rstrip())
        elif line.startswith("define"):
            name = next((c for c in reversed(pending)
                         if not c.startswith("Function Attrs")), "<unknown>")
            body, j = [line], i + 1
            while j < n and not lines[j].startswith("}"):
                body.append(lines[j]);
                j += 1
            match = next((t for t in TARGETS if t in name), None)
            if match and match not in found:
                allocas = [b.strip() for b in body if "alloca" in b]
                heaps = [b.strip() for b in body if HEAP_RE.search(b)]
                found[match] = dict(
                    mangled=name,
                    alloca_count=len(allocas),
                    stack_bytes=sum(int(m.group(1)) for b in allocas
                                    if (m := ARRAY_RE.search(b))),
                    heap_count=len(heaps),
                    heap_bytes=sum(heap_bytes(b) for b in heaps),
                )
                for a in allocas:
                    m = ARRAY_RE.search(a)
                    raw_rows.append(dict(func=match, kind="stack",
                                         bytes=int(m.group(1)) if m else None,
                                         ir=a))
                for h in heaps:
                    raw_rows.append(dict(func=match, kind="heap",
                                         bytes=heap_bytes(h) or None,
                                         ir=h))
            pending, i = [], j
        i += 1

    summary = []
    for grp, op, name in SPEC:
        d = found.get(name)
        summary.append(dict(
            group=grp, op=op, func=name, present=d is not None,
            alloca_count=d["alloca_count"] if d else 0,
            stack_bytes=d["stack_bytes"] if d else 0,
            heap_count=d["heap_count"] if d else 0,
            heap_bytes=d["heap_bytes"] if d else 0,
            mangled=d["mangled"] if d else None,
        ))
    df = pd.DataFrame(summary)
    raw = pd.DataFrame(raw_rows, columns=["func", "kind", "bytes", "ir"])
    return df, raw


In [ ]:
dfs, raws = [], []
for (arch, opt), path in FILES.items():
    d, r = parse_ir(path)
    d.insert(0, "arch", arch);
    d.insert(1, "opt", opt)
    r.insert(0, "arch", arch);
    r.insert(1, "opt", opt)
    dfs.append(d);
    raws.append(r)

df = pd.concat(dfs, ignore_index=True)
raw = pd.concat(raws, ignore_index=True)

df["arch"] = pd.Categorical(df["arch"], categories=["x86", "aarch64"], ordered=True)
df["opt"] = pd.Categorical(df["opt"], categories=["o0", "o3"], ordered=True)
df["group"] = pd.Categorical(df["group"],
                             categories=["scalar", "sse2", "avx2", "avx512", "neon"], ordered=True)
df["op"] = pd.Categorical(df["op"],
                          categories=["encrypt", "encrypt_inflight", "decrypt"], ordered=True)
df = df.sort_values(["arch", "opt", "group", "op"]).reset_index(drop=True)


In [ ]:
present = df[df.present].drop(columns=["present", "mangled"])
present


In [ ]:
present.pivot_table(
    index=["group", "op"], columns=["arch", "opt"],
    values="stack_bytes", observed=True
)


In [ ]:
present.groupby(["arch", "opt", "group"], observed=True)[
    ["alloca_count", "stack_bytes", "heap_count", "heap_bytes"]].sum()


In [ ]:
backend_map = {
    "scalar": "skalarny",
    "sse2": "SSE2",
    "avx2": "AVX2",
    "avx512": "AVX-512",
    "neon": "NEON",
}

op_map = {
    "encrypt": "Szyfrowanie",
    "encrypt_inflight": "Szyfr. w locie",
    "decrypt": "Deszyfrowanie",
}

arch_map = {
    "x86": "x86\_64",
}

In [ ]:
table = (present.groupby(["arch", "opt", "group", "op"], observed=True)[
             ["alloca_count", "stack_bytes", "heap_count", "heap_bytes"]].sum()
         .unstack("opt")
         .swaplevel(axis=1)
         .sort_index(axis=1))

table = table.drop(columns=["heap_count", "heap_bytes"], level=1)

table = table.rename(index=arch_map, level="arch")
table = table.rename(index=backend_map, level="group")
table = table.rename(index=op_map, level="op")

os.makedirs("../csv", exist_ok=True)

out = table.copy()
out.columns = ["_".join(map(str, c)) for c in out.columns]
out = out.reset_index()
out.to_csv("../csv/memory_results.csv", index=False)

In [ ]:
def print_raw(arch, opt, kinds=("stack", "heap")):
    sub = raw[(raw.arch == arch) & (raw.opt == opt)]
    print("=" * 70)
    print(f" {arch}  {opt}")
    print("=" * 70)
    for grp, op, name in SPEC:
        d = df[(df.arch == arch) & (df.opt == opt) & (df.func == name)]
        if d.empty or not bool(d.present.iloc[0]):
            continue
        r0 = d.iloc[0]
        rows = sub[sub.func == name]
        print(f"\n=== [{grp}/{op}] {r0.mangled}")
        if "stack" in kinds:
            print(f"   stos:   {r0.alloca_count} alloca, >= {r0.stack_bytes} B")
            for ir in rows[rows.kind == "stack"].ir:
                print(f"            {ir}")
        if "heap" in kinds:
            print(f"   sterta: {r0.heap_count} wywolan, >= {r0.heap_bytes} B")
            for ir in rows[rows.kind == "heap"].ir:
                print(f"            {ir}")


In [ ]:
print_raw("x86", "o0")

In [ ]:
print_raw("x86", "o3")

In [ ]:
print_raw("aarch64", "o0")

In [ ]:
print_raw("aarch64", "o3")